# Assignment 1 — Pandas Practice Project

Build a small sales analytics report step by step. Each **TASK** is broken into
small cells — run them one at a time and discuss the output as we go.

Dataset: `datasets/sales.csv` (+ `datasets/products.json` for the bonus).

In [1]:
import os
import pandas as pd

In [2]:
# resolve datasets/ no matter where Jupyter is launched from
DATASETS = next(
    os.path.join(p, "datasets")
    for p in [".", "..", "../..", "../../.."]
    if os.path.isdir(os.path.join(p, "datasets"))
)
SALES = os.path.join(DATASETS, "sales.csv")
print("datasets at", DATASETS)

datasets at ../../datasets


### TASK 1 — Load `sales.csv` into `df`, parsing `date` as datetime.

In [3]:
df = pd.read_csv(SALES, parse_dates=["date"])

In [4]:
assert df.shape == (54, 7), "Expected 54 rows x 7 columns"
print(f"loaded {df.shape[0]} rows, {df.shape[1]} columns")

loaded 54 rows, 7 columns


In [5]:
df.head()

,date,product,category,quantity,unit_price,revenue,region
0,2024-01-02,Laptop Pro,Electronics,3,35000,105000,Bangkok
1,2024-01-02,Wireless Mouse,Electronics,15,650,9750,Bangkok
2,2024-01-03,Standing Desk,Furniture,2,8500,17000,Chiang Mai
3,2024-01-03,Ergonomic Chair,Furniture,4,15000,60000,Bangkok
4,2024-01-04,Running Shoes,Clothing,10,2800,28000,Phuket


### TASK 2 — Add a `month` column (`YYYY-MM`) from `date`.

In [6]:
df["month"] = df["date"].dt.strftime("%Y-%m")

In [7]:
sorted(df["month"].unique())

['2024-01', '2024-02', '2024-03']

### TASK 3 — Total revenue per category, sorted high → low.

In [8]:
by_category = df.groupby("category")["revenue"].sum()

In [9]:
by_category.sort_values(ascending=False)

category
Electronics    1408500
Furniture       442600
Clothing        292100
Accessories     108150
Name: revenue, dtype: int64

### TASK 4 — Top 5 products by total revenue.

In [10]:
rev_by_product = df.groupby("product")["revenue"].sum()

In [11]:
rev_by_product.sort_values(ascending=False).head(5)

product
Laptop Pro         805000
Ergonomic Chair    300000
Monitor 27in       300000
Running Shoes      176400
Webcam             157500
Name: revenue, dtype: int64

### TASK 5 — Pivot: revenue by region (rows) × category (columns).

In [12]:
pivot = df.pivot_table(
    values="revenue", index="region", columns="category", aggfunc="sum", fill_value=0
)

In [13]:
pivot.astype(int)

category,Accessories,Clothing,Electronics,Furniture
region,,,,
Ayutthaya,16800,0,13000,19200
Bangkok,30000,99350,879750,295500
Chiang Mai,6000,73150,224250,55400
Khon Kaen,23850,35600,99000,8500
Phuket,31500,84000,192500,64000


### BONUS — Merge with `products.json` to get supplier, then revenue per supplier.

In [14]:
products = pd.json_normalize(pd.read_json(os.path.join(DATASETS, "products.json"))["products"])

In [15]:
enriched = df.merge(products[["name", "supplier"]], left_on="product", right_on="name", how="left")

In [16]:
enriched.groupby("supplier")["revenue"].sum().sort_values(ascending=False).astype(int)

supplier
TechCorp        962500
FurniturePro    442600
DisplayTech     300000
SportWear       292100
PeripheralCo    146000
OfficePro        56400
HealthGear       51750
Name: revenue, dtype: int64

**Done!** Try changing a step and predict how the output moves.